In [1]:
"""
4.4 Controller Behavior Analysis
=================================
분석 대상:
  - results/{model}/{dataset}/{strategy}/trajectory_logs.jsonl
  - results/{model}/{dataset}/{strategy}/step_logs.jsonl
  - results/{model}/{dataset}/{strategy}/summary.json

전략: repair_loop, code_then_plan, code_then_plan_repair, policy_loop
모델: phi35mini, qwen25coder7b (또는 실제 폴더명)
데이터셋: humaneval, mbpp
"""

from __future__ import annotations

import json
import os
from collections import defaultdict
from pathlib import Path
from typing import Optional

import pandas as pd

# ─────────────────────────────────────────────────────────
# 0. 경로 설정
# ─────────────────────────────────────────────────────────

RESULTS_ROOT = Path("../results")   # 실제 결과 루트 경로로 변경하세요

MODELS    = ["phi35mini", "qwen25coder7b"]   # 실제 폴더명으로 수정
DATASETS  = ["humaneval", "mbpp"]
STRATEGIES = ["repair_loop", "code_then_plan", "code_then_plan_repair", "policy_loop"]


# ─────────────────────────────────────────────────────────
# 1. 데이터 로더
# ─────────────────────────────────────────────────────────

def load_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_json(path: Path) -> dict:
    if not path.exists():
        return {}
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def collect_all_data() -> tuple[pd.DataFrame, pd.DataFrame, list[dict]]:
    """
    Returns:
        traj_df  : trajectory_logs 전체
        step_df  : step_logs 전체
        summaries: summary.json 리스트 (meta 포함)
    """
    traj_rows, step_rows, summaries = [], [], []

    for model in MODELS:
        for dataset in DATASETS:
            for strategy in STRATEGIES:
                base = RESULTS_ROOT / model / dataset / strategy

                # trajectory_logs
                for row in load_jsonl(base / "trajectory_logs.jsonl"):
                    row["_model"] = model
                    row["_dataset"] = dataset
                    row["_strategy"] = strategy
                    traj_rows.append(row)

                # step_logs
                for row in load_jsonl(base / "step_logs.jsonl"):
                    row["_model"] = model
                    row["_dataset"] = dataset
                    row["_strategy"] = strategy
                    step_rows.append(row)

                # summary
                s = load_json(base / "summary.json")
                if s:
                    s["_model"] = model
                    s["_dataset"] = dataset
                    s["_strategy"] = strategy
                    summaries.append(s)

    traj_df = pd.DataFrame(traj_rows) if traj_rows else pd.DataFrame()
    step_df = pd.DataFrame(step_rows) if step_rows else pd.DataFrame()
    return traj_df, step_df, summaries


# ─────────────────────────────────────────────────────────
# 2. (1) Action Selection Statistics
# ─────────────────────────────────────────────────────────

def analysis_action_selection(traj_df: pd.DataFrame, step_df: pd.DataFrame):
    print("\n" + "="*60)
    print("(1) Action Selection Statistics")
    print("="*60)

    if step_df.empty:
        print("  [데이터 없음]")
        return

    # ── 2-1. 전략별 · 모델별 · 데이터셋별 stage 사용 분포
    print("\n[1-A] stage 사용 분포 (전략 × 모델 × 데이터셋)")
    stage_cols = ["_strategy", "_model", "_dataset", "stage"]
    if all(c in step_df.columns for c in stage_cols):
        stage_dist = (
            step_df.groupby(stage_cols)
            .size()
            .rename("count")
            .reset_index()
        )
        print(stage_dist.to_string(index=False))

    # ── 2-2. failure family별 다음 action 분포 (policy_loop)
    print("\n[1-B] failure_family별 policy_action 분포 (policy_loop only)")
    pl_steps = step_df[step_df["_strategy"] == "policy_loop"].copy() if "_strategy" in step_df.columns else pd.DataFrame()

    if not pl_steps.empty and "policy_action" in pl_steps.columns and "status" in pl_steps.columns:
        pl_steps["failure_family"] = pl_steps["status"].apply(
            lambda s: "PASS" if s == "PASS"
            else ("PLAN_DONE" if s == "PLAN_DONE"
                  else str(s).split(":")[0])
        )
        fa_action = (
            pl_steps[pl_steps["failure_family"] != "PASS"]
            .groupby(["_model", "_dataset", "failure_family", "policy_action"])
            .size()
            .rename("count")
            .reset_index()
        )
        print(fa_action.to_string(index=False))
    else:
        print("  [policy_loop step_logs 없음 또는 policy_action 필드 없음]")

    # ── 2-3. error_type별 다음 action (policy_loop — rule 검증)
    print("\n[1-C] error_type × policy_action (policy_loop rule 검증)")
    if not pl_steps.empty and "error_type" in pl_steps.columns and "policy_action" in pl_steps.columns:
        et_action = (
            pl_steps[
                pl_steps["stage"].isin(["repair", "plan", "plan_code"])
            ]
            .groupby(["_model", "_dataset", "error_type", "policy_action"])
            .size()
            .rename("count")
            .reset_index()
        )
        print(et_action.to_string(index=False))


# ─────────────────────────────────────────────────────────
# 3. (2) Planning Escalation Analysis
# ─────────────────────────────────────────────────────────

def analysis_planning_escalation(traj_df: pd.DataFrame, step_df: pd.DataFrame):
    print("\n" + "="*60)
    print("(2) Planning Escalation Analysis")
    print("="*60)

    if traj_df.empty:
        print("  [데이터 없음]")
        return

    escalation_strategies = ["policy_loop", "code_then_plan", "code_then_plan_repair"]
    esc_df = traj_df[traj_df["_strategy"].isin(escalation_strategies)].copy() if "_strategy" in traj_df.columns else pd.DataFrame()

    if esc_df.empty:
        print("  [escalation 전략 데이터 없음]")
        return

    # ── 3-1. planning cycle 사용 빈도 분포
    print("\n[2-A] planning_cycle_count 분포 (전략 × 모델 × 데이터셋)")
    if "planning_cycle_count" in esc_df.columns:
        cycle_dist = (
            esc_df.groupby(["_strategy", "_model", "_dataset"])["planning_cycle_count"]
            .value_counts()
            .rename("n_trajectories")
            .reset_index()
            .sort_values(["_strategy", "_model", "_dataset", "planning_cycle_count"])
        )
        print(cycle_dist.to_string(index=False))

    # ── 3-2. plan escalation → recovery 성공률
    print("\n[2-B] plan escalation 사용 vs 성공률")
    if "used_plan" in esc_df.columns and "final_status" in esc_df.columns:
        esc_df["success"] = esc_df["final_status"] == "PASS"
        plan_recovery = (
            esc_df.groupby(["_strategy", "_model", "_dataset", "used_plan"])["success"]
            .agg(["sum", "count"])
            .rename(columns={"sum": "success", "count": "total"})
            .assign(success_rate=lambda x: x["success"] / x["total"])
            .reset_index()
        )
        print(plan_recovery.to_string(index=False))

    # ── 3-3. recovered_by 분포
    print("\n[2-C] recovered_by 분포")
    if "recovered_by" in esc_df.columns:
        rec_dist = (
            esc_df.groupby(["_strategy", "_model", "_dataset", "recovered_by"])
            .size()
            .rename("count")
            .reset_index()
        )
        print(rec_dist.to_string(index=False))

    # ── 3-4. policy_loop stop_reason 분포
    print("\n[2-D] stop_reason 분포 (policy_loop)")
    pl_traj = esc_df[esc_df["_strategy"] == "policy_loop"]
    if not pl_traj.empty and "stop_reason" in pl_traj.columns:
        sr_dist = (
            pl_traj.groupby(["_model", "_dataset", "stop_reason"])
            .size()
            .rename("count")
            .reset_index()
        )
        print(sr_dist.to_string(index=False))

    # ── 3-5. stagnation 후 plan escalation (transition_path 분석)
    print("\n[2-E] repair 연속 실패 후 plan 전환 패턴 (policy_loop)")
    pl_traj2 = traj_df[traj_df["_strategy"] == "policy_loop"].copy() if "_strategy" in traj_df.columns else pd.DataFrame()
    if not pl_traj2.empty and "transition_path" in pl_traj2.columns and "planning_cycle_count" in pl_traj2.columns:
        # repair를 1회 이상 쓰고 plan으로 escalation한 케이스
        escalated = pl_traj2[
            (pl_traj2["repair_call_count"] > 0) &
            (pl_traj2["planning_cycle_count"] > 0)
        ]
        total_failed = pl_traj2[pl_traj2["final_status"] != "PASS"]
        print(f"  repair 사용 후 plan escalation: {len(escalated)} trajectories")
        grp = escalated.groupby(["_model", "_dataset"]).size().rename("escalated_count").reset_index()
        print(grp.to_string(index=False))


# ─────────────────────────────────────────────────────────
# 4. (3) Budget Efficiency Analysis
# ─────────────────────────────────────────────────────────

def analysis_budget_efficiency(traj_df: pd.DataFrame, step_df: pd.DataFrame, summaries: list[dict]):
    print("\n" + "="*60)
    print("(3) Budget Efficiency Analysis")
    print("="*60)

    # ── 4-1. 평균 call 수
    print("\n[3-A] 평균 call 수 (전략 × 모델 × 데이터셋)")
    if not traj_df.empty and "call_count" in traj_df.columns:
        avg_calls = (
            traj_df.groupby(["_strategy", "_model", "_dataset"])["call_count"]
            .agg(["mean", "median", "std", "count"])
            .round(3)
            .reset_index()
        )
        print(avg_calls.to_string(index=False))

    # summary.json의 avg_calls도 출력 (cross-check)
    if summaries:
        print("\n  [summary.json avg_calls 교차 검증]")
        for s in summaries:
            if "avg_calls" in s:
                print(f"  {s.get('_strategy','?')} / {s.get('_model','?')} / {s.get('_dataset','?')}: avg_calls={s['avg_calls']:.3f}")

    # ── 4-2. Early success rate (1-shot: generate 단계에서 바로 PASS)
    print("\n[3-B] Early success rate (generate 단계 1-shot PASS)")
    if not step_df.empty and "stage" in step_df.columns and "status" in step_df.columns:
        gen_steps = step_df[step_df["stage"] == "generate"].copy()
        gen_steps["early_success"] = gen_steps["status"] == "PASS"
        early_sr = (
            gen_steps.groupby(["_strategy", "_model", "_dataset"])["early_success"]
            .agg(["sum", "count"])
            .rename(columns={"sum": "early_success", "count": "total"})
            .assign(early_success_rate=lambda x: (x["early_success"] / x["total"]).round(4))
            .reset_index()
        )
        print(early_sr.to_string(index=False))

    # ── 4-3. Recovery speed: 첫 실패 후 PASS까지 추가 call 수
    print("\n[3-C] Recovery speed (첫 실패 문제에서 PASS까지 소요 call 수)")
    if not traj_df.empty and "call_count" in traj_df.columns and "final_status" in traj_df.columns:
        # 첫 generate에서 실패했지만 최종 PASS한 케이스
        recovered = traj_df[
            (traj_df["final_status"] == "PASS") &
            (traj_df["call_count"] > 1)
        ].copy()
        if not recovered.empty:
            # 첫 generate 이후 추가 call = call_count - 1
            recovered["recovery_calls"] = recovered["call_count"] - 1
            rec_speed = (
                recovered.groupby(["_strategy", "_model", "_dataset"])["recovery_calls"]
                .agg(["mean", "median", "min", "max", "count"])
                .round(3)
                .reset_index()
            )
            print(rec_speed.to_string(index=False))
        else:
            print("  [recovery case 없음]")

    # ── 4-4. 전략별 성공률 종합
    print("\n[3-D] 전략별 최종 성공률 (success@k)")
    if summaries:
        rows = []
        for s in summaries:
            k = s.get("max_calls", "?")
            key = f"success@{k}" if isinstance(k, int) else "success_at_k"
            rows.append({
                "strategy": s.get("_strategy", "?"),
                "model":    s.get("_model", "?"),
                "dataset":  s.get("_dataset", "?"),
                "max_calls": k,
                "success@k": s.get(key, s.get("success_at_k", None)),
                "ausc":      s.get("ausc", None),
                "exec_success_rate": s.get("execution_success_rate", None),
            })
        sr_df = pd.DataFrame(rows)
        print(sr_df.to_string(index=False))

    # ── 4-5. token 효율성
    print("\n[3-E] 평균 총 token 사용량 (전략 × 모델 × 데이터셋)")
    if not traj_df.empty and "total_tokens" in traj_df.columns:
        token_eff = (
            traj_df.groupby(["_strategy", "_model", "_dataset"])["total_tokens"]
            .agg(["mean", "median"])
            .round(1)
            .reset_index()
        )
        print(token_eff.to_string(index=False))

    # ── 4-6. plan call 비용 대비 효과
    print("\n[3-F] plan call 비용 대비 효과 (planning 전략만)")
    plan_strategies = ["code_then_plan", "code_then_plan_repair", "policy_loop"]
    if not traj_df.empty and "_strategy" in traj_df.columns:
        plan_df = traj_df[traj_df["_strategy"].isin(plan_strategies)].copy()
        if not plan_df.empty and "planning_cycle_count" in plan_df.columns:
            plan_df["success"] = plan_df["final_status"] == "PASS"
            plan_df["plan_used"] = plan_df["planning_cycle_count"] > 0
            plan_eff = (
                plan_df.groupby(["_strategy", "_model", "_dataset", "plan_used"])
                .agg(
                    n=("success", "count"),
                    success_rate=("success", "mean"),
                    avg_plan_cycles=("planning_cycle_count", "mean"),
                    avg_calls=("call_count", "mean"),
                )
                .round(3)
                .reset_index()
            )
            print(plan_eff.to_string(index=False))


# ─────────────────────────────────────────────────────────
# 5. Main
# ─────────────────────────────────────────────────────────

def main():
    print("데이터 로딩 중...")
    traj_df, step_df, summaries = collect_all_data()

    print(f"  trajectory_logs: {len(traj_df)} rows")
    print(f"  step_logs      : {len(step_df)} rows")
    print(f"  summaries      : {len(summaries)} files")

    if traj_df.empty and step_df.empty:
        print("\n⚠️  데이터가 없습니다. RESULTS_ROOT 경로와 MODELS/DATASETS/STRATEGIES 설정을 확인하세요.")
        print(f"  현재 RESULTS_ROOT: {RESULTS_ROOT.resolve()}")
        return

    analysis_action_selection(traj_df, step_df)
    analysis_planning_escalation(traj_df, step_df)
    analysis_budget_efficiency(traj_df, step_df, summaries)

    print("\n" + "="*60)
    print("✅ 분석 완료")
    print("="*60)


if __name__ == "__main__":
    main()

데이터 로딩 중...
  trajectory_logs: 1684 rows
  step_logs      : 9086 rows
  summaries      : 8 files

(1) Action Selection Statistics

[1-A] stage 사용 분포 (전략 × 모델 × 데이터셋)
            _strategy        _model  _dataset     stage  count
       code_then_plan     phi35mini humaneval  generate    164
       code_then_plan     phi35mini humaneval      plan    567
       code_then_plan     phi35mini humaneval plan_code    567
       code_then_plan     phi35mini      mbpp  generate    257
       code_then_plan     phi35mini      mbpp      plan    593
       code_then_plan     phi35mini      mbpp plan_code    593
       code_then_plan qwen25coder7b humaneval  generate    164
       code_then_plan qwen25coder7b humaneval      plan    164
       code_then_plan qwen25coder7b humaneval plan_code    164
       code_then_plan qwen25coder7b      mbpp  generate    257
       code_then_plan qwen25coder7b      mbpp      plan    493
       code_then_plan qwen25coder7b      mbpp plan_code    493
code_then_plan_